In [1]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG (FIX XUNG ĐỘT PHIÊN BẢN) ──────────────────────────
!pip uninstall -y peft accelerate transformers datasets numpy pyarrow adapters -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.20.0 \
  safetensors==0.4.2 \
  scikit-learn \
  Pillow \
  "numpy<2.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

In [ ]:


# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools, json
import torch
import numpy as np
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset, Image as HFImage
from google.colab import drive

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_Adapter_IFND_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")

# Ép kiểu dữ liệu ảnh để tránh lỗi ảnh Grayscale
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=True))

model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))

    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]
print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 4. MODEL ARCHITECTURE (MANUAL ADAPTER + GATED FUSION + ALIGNMENT) ────────
class Adapter(nn.Module):
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, bottleneck),
            nn.GELU(),
            nn.Linear(bottleneck, dim)
        )
    def forward(self, x):
        return x + self.net(x)

class CLIPWithAdapter(nn.Module):
    def __init__(self, model_name, num_labels=2, bottleneck=64, lambda_weight=0.1, delta=0.5):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        embed_dim = self.clip.config.projection_dim # 512

        # 1. Đóng băng Base Model
        for param in self.clip.parameters():
            param.requires_grad = False

        text_dim = self.clip.config.text_config.hidden_size
        img_dim  = self.clip.config.vision_config.hidden_size

        # 2. Khởi tạo Manual Adapters
        self.text_adapters = nn.ModuleList([Adapter(text_dim, bottleneck) for _ in range(self.clip.config.text_config.num_hidden_layers)])
        self.img_adapters = nn.ModuleList([Adapter(img_dim, bottleneck) for _ in range(self.clip.config.vision_config.num_hidden_layers)])

        # 3. Gated Fusion & Alignment Head (Mở khóa gradient)
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        for param in self.cross_attention.parameters(): param.requires_grad = True
        for param in self.W_g.parameters(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        # --- Xử lý Text ---
        text_outputs = self.clip.text_model(
            input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True, return_dict=True
        )
        text_hidden = text_outputs.last_hidden_state
        for adapter in self.text_adapters:
            text_hidden = adapter(text_hidden)

        text_embeds = self.clip.text_projection(text_hidden[torch.arange(text_hidden.shape[0]), input_ids.argmax(dim=-1)])
        h_T = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # --- Xử lý Image ---
        vision_outputs = self.clip.vision_model(
            pixel_values=pixel_values, output_hidden_states=True, return_dict=True
        )
        img_hidden = vision_outputs.last_hidden_state
        for adapter in self.img_adapters:
            img_hidden = adapter(img_hidden)

        img_embeds = self.clip.visual_projection(img_hidden[:, 0, :])
        h_I = img_embeds / img_embeds.norm(dim=-1, keepdim=True)

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)
        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- CLASSIFICATION & ALIGNMENT LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)

            # TƯƠNG ĐỒNG TUYỆT ĐỐI VỚI CÁC BẢN TRƯỚC ĐỂ ĐỐI CHỨNG
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)

            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()
model = CLIPWithAdapter(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4, # Giữ mức 1e-4 vì kiến trúc Adapter cần LR cao hơn FFT/LoRA
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Bắt đầu huấn luyện Adapter...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()
trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60
print(f"✅ Training hoàn tất: {train_time_min:.2f} phút")

# ─── 6. INFERENCE MEASUREMENT (BATCH SIZE = 1, P50 MEDIAN) ────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "Sample news for latency measurement."
inputs = processor(text=dummy_text, images=dummy_image, truncation=True, padding="max_length", max_length=77, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values = inputs["pixel_values"].to(device)

if device == "cuda": torch.cuda.synchronize()

# Warmup
with torch.no_grad():
    for _ in range(50): model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

# ─── 7. FINAL EVALUATION & SAVE RESULTS ───────────────────────────────────────
print("\n📊 Evaluating on IFND TEST split...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs[:, 1])

final_results = {
    "Model": "CLIP ViT-B/32",
    "Method": "Adapter (Manual Bottleneck=64)",
    "Dataset": "IFND-multimodal",
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Trainable_Params_M": round(trainable_params / 1e6, 2),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(vram, 2),
        "Latency_P50_ms": round(np.median(latencies), 2),
        "Latency_P90_ms": round(np.percentile(latencies, 90), 2)
    },
    "Test_Metrics": {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(p * 100, 2),
        "Recall": round(r * 100, 2),
        "F1_Score": round(f1 * 100, 2),
        "AUC": round(auc, 4)
    }
}

json_path = os.path.join(SAVE_DIR, "results_Adapter_IFND.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\n[TEST RESULT]")
print(f" Accuracy : {acc*100:.2f}%")
print(f" Precision: {p*100:.2f}%")
print(f" Recall   : {r*100:.2f}%")
print(f" F1 Score : {f1*100:.2f}% | AUC: {auc:.4f}")
print(f"\n💾 Đã lưu toàn bộ kết quả vào file: {json_path}")
print(f"{'='*50}")

# ─── 8. AUTO-DISCONNECT GPU ───────────────────────────────────────────────────
print("⏳ Đang ngắt kết nối GPU...")
from google.colab import runtime
time.sleep(5)
runtime.unassign()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_ViT_B32_Adapter_IFND_Final
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

...


📊 Total params    : 154,836,739 (~154.84M)
📊 Trainable params: 3,559,426 (2.30%)

🚀 Bắt đầu huấn luyện Adapter...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.454900,0.105714,0.980989,0.979268,0.996278,0.987700,0.993640
2,0.069300,0.066253,0.986692,0.991315,0.991315,0.991315,0.996338
3,0.037200,0.058550,0.986692,0.986486,0.996278,0.991358,0.996906
4,0.028800,0.058725,0.988593,0.988916,0.996278,0.992583,0.996881
5,0.026000,0.054770,0.989544,0.990136,0.996278,0.993197,0.996704


✅ Training hoàn tất: 13.39 phút

📊 Evaluating on IFND TEST split...



[TEST RESULT]
 Accuracy : 99.15%
 Precision: 99.14%
 Recall   : 99.75%
 F1 Score : 99.44% | AUC: 0.9965

💾 Đã lưu toàn bộ kết quả vào file: /content/drive/MyDrive/CLIP_ViT_B32_Adapter_IFND_Final/results_Adapter_IFND.json
⏳ Đang ngắt kết nối GPU...
